# Nhóm 2 — CNN + MLP trên Kaggle
Notebook dùng chung lõi `src/` và runner, ưu tiên CUDA (P100/T4x2 nếu runtime cấp), seed cố định 42.

In [ ]:
from pathlib import Path
import os, sys, torch
PROJECT_ROOT=Path('/kaggle/working/BTL03'); INPUT_ROOT=Path('/kaggle/input'); WORK_ROOT=Path('/kaggle/working')
if not (PROJECT_ROOT/'scripts/run_group2_cnn.py').is_file(): raise FileNotFoundError('Checkout BTL03 vào /kaggle/working/BTL03.')
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Input mounts:',[p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else [])

## Dataset và thực nghiệm
Gắn MNIST/WIKI/FER vào `/kaggle/input/` hoặc đặt/copy đúng file theo YAML. Không dùng output của smoke-test làm kết quả nghiệm thu.

In [ ]:
import shutil
# Tìm đúng tên file trong dataset Kaggle đã attach; không tải/di chuyển dữ liệu nguồn.
expected={'data/raw/mnist/train_images.npy':'train_images.npy','data/raw/mnist/train_labels.npy':'train_labels.npy','data/raw/mnist/test_images.npy':'test_images.npy','data/raw/mnist/test_labels.npy':'test_labels.npy','data/raw/wiki_gender/train_x.npy':'train_x.npy','data/raw/wiki_gender/train_y.npy':'train_y.npy','data/raw/wiki_gender/validation_x.npy':'validation_x.npy','data/raw/wiki_gender/validation_y.npy':'validation_y.npy','data/raw/fer2013/fer2013_training_onehot.csv':'fer2013_training_onehot.csv','data/raw/fer2013/fer2013_publictest_onehot.csv':'fer2013_publictest_onehot.csv'}
for destination_name, basename in expected.items():
    matches=list(INPUT_ROOT.rglob(basename)) if INPUT_ROOT.exists() else []
    if matches:
        destination=PROJECT_ROOT/destination_name; destination.parent.mkdir(parents=True,exist_ok=True)
        if not destination.exists(): shutil.copy2(matches[0],destination)
print('Đã ánh xạ các file dataset tìm thấy từ Kaggle input.')
!python scripts/download_data.py check

In [ ]:
!python scripts/run_group2_cnn.py --model all

In [ ]:
import pandas as pd
from IPython.display import display
history=pd.read_csv(PROJECT_ROOT/'experiments/csv_logs/EmotionCNN_history.csv')
display(history)
display(history.plot(x='Epoch',y=['Train_Loss','Val_Loss'],title='FER2013 baseline overfitting').figure)

## Đóng gói artifacts
ZIP chỉ chứa artifacts được tạo (checkpoint, features, CSV, TensorBoard và hình); dữ liệu nguồn không được đóng gói mặc định.

In [ ]:
import shutil
artifact=WORK_ROOT/'group2_cnn_artifacts'
artifact.mkdir(exist_ok=True)
for relative in ['outputs/checkpoints/02_cnn_mlp','outputs/features/mnist_cnn_pool3_features.npz','experiments/csv_logs','experiments/tensorboard/Group2_CNN','outputs/figures']:
    source=PROJECT_ROOT/relative
    if source.exists():
        target=artifact/relative.replace('/','_')
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
zip_path=shutil.make_archive(str(WORK_ROOT/'group2_cnn_artifacts'),'zip',artifact)
print('Artifact ZIP:',zip_path)